In [0]:
# ==========================================
# 02_silver_clean — Bronze -> Silver
# ==========================================

# ----------------------------
# Standard widgets (all layers)
# ----------------------------
dbutils.widgets.text("base_path", "/Volumes/workspace/ecommerce/ecommerce_data")
dbutils.widgets.text("raw_path", "/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv")  # not used here
dbutils.widgets.text("bronze_path", "")
dbutils.widgets.text("silver_path", "")
dbutils.widgets.text("gold_path", "")  # not used here
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "full"])

base_path = dbutils.widgets.get("base_path").strip()
bronze_path = dbutils.widgets.get("bronze_path").strip()
silver_path = dbutils.widgets.get("silver_path").strip()
mode = dbutils.widgets.get("mode").strip()
#mode = "full" # for start 

if not bronze_path:
    bronze_path = f"{base_path}/bronze/events_raw"
if not silver_path:
    silver_path = f"{base_path}/silver/events"

print(f"bronze_path={bronze_path}")
print(f"silver_path={silver_path}")
print(f"mode={mode}")

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Delta merge support
from delta.tables import DeltaTable


# ----------------------------
# Helpers
# ----------------------------
def delta_path_exists(path: str) -> bool:
    # A simple existence check in Databricks FS
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

def require(value, name):
    if value is None or str(value).strip() == "":
        raise ValueError(f"Missing required parameter: {name}")
    return value

require(bronze_path, "bronze_path")
require(silver_path, "silver_path")


# ----------------------------
# Read Bronze
# ----------------------------
bronze_df = spark.read.format("delta").load(bronze_path)

# bronze_df expected columns (from your pipeline):
# event_time, event_type, product_id, category_id, category_code, brand, price, user_id, user_session
# ingestion_ts, ingestion_date, source_file (optional)

# ----------------------------
# Incremental filter strategy
# ----------------------------
# If silver exists, only process bronze records with ingestion_ts greater than max processing_ts in silver (best-effort),
# otherwise process all.
# This relies on us writing processing_ts in silver.
if mode == "incremental" and delta_path_exists(silver_path):
    silver_existing = spark.read.format("delta").load(silver_path)

    # If processing_ts doesn't exist yet (older table), fallback to ingestion_ts
    if "processing_ts" in silver_existing.columns:
        watermark_col = "processing_ts"
    elif "ingestion_ts" in silver_existing.columns:
        watermark_col = "ingestion_ts"
    else:
        watermark_col = None

    if watermark_col:
        max_ts = silver_existing.select(F.max(F.col(watermark_col)).alias("mx")).collect()[0]["mx"]
        if max_ts is not None:
            bronze_df = bronze_df.filter(F.col("ingestion_ts") > F.lit(max_ts))
            print(f"Incremental mode: filtering bronze to ingestion_ts > max({watermark_col})={max_ts}")
        else:
            print(f"Incremental mode: {watermark_col} max is null; processing all bronze.")
    else:
        print("Incremental mode: no watermark column found in silver; processing all bronze.")
else:
    if mode == "incremental":
        print("Incremental mode: silver does not exist yet; processing all bronze.")
    else:
        print("Full mode: processing all bronze.")


# ----------------------------
# Clean + standardise (Silver logic)
# ----------------------------
# 1) Parse event_time (dataset often uses 'yyyy-MM-dd HH:mm:ss' as string)
# 2) Cast ids and price
# 3) Normalise strings
# 4) Drop clearly invalid rows
# 5) De-duplicate within batch
silver_batch = (
    bronze_df
    .withColumn("event_ts", F.to_timestamp("event_time"))  # robust; returns null if parse fails
    .withColumn("event_date", F.to_date("event_ts"))
    .withColumn("event_type", F.lower(F.trim(F.col("event_type"))))
    .withColumn("category_code", F.lower(F.trim(F.col("category_code"))))
    .withColumn("brand", F.lower(F.trim(F.col("brand"))))
    .withColumn("product_id", F.col("product_id").cast("long"))
    .withColumn("category_id", F.col("category_id").cast("long"))
    .withColumn("user_id", F.col("user_id").cast("long"))
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("processing_ts", F.current_timestamp())
)

# Basic validity rules (tune as needed)
silver_batch = silver_batch.filter(
    (F.col("event_ts").isNotNull()) &
    (F.col("event_type").isin("view", "cart", "purchase")) &
    (F.col("product_id").isNotNull()) &
    (F.col("user_session").isNotNull()) &
    (F.col("event_date").isNotNull()) &
    (F.col("price").isNotNull()) &
    (F.col("price") >= 0)
)

# De-dup within the incoming batch: keep latest ingestion_ts if duplicates
dedup_keys = ["user_session", "event_ts", "event_type", "product_id"]
w = Window.partitionBy(*dedup_keys).orderBy(F.col("ingestion_ts").desc_nulls_last(), F.col("processing_ts").desc())
silver_batch = (
    silver_batch
    .withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

# Select a clean, stable column set
# Keep the original event_time string if you want traceability
final_cols = [
    "event_time", "event_ts", "event_date", "event_type",
    "product_id", "category_id", "category_code", "brand",
    "price", "user_id", "user_session",
    "ingestion_ts", "ingestion_date", "source_file",
    "processing_ts"
]

# Some bronze runs might not have source_file; select what exists
available_cols = set(silver_batch.columns)
final_cols = [c for c in final_cols if c in available_cols]

silver_batch = silver_batch.select(*final_cols)

batch_count = silver_batch.count()
print(f"Silver batch rows after cleaning = {batch_count}")

if batch_count == 0:
    # Still set task value to avoid downstream confusion
    dbutils.jobs.taskValues.set(key="impacted_dates", value="")
    dbutils.jobs.taskValues.set(key="silver_batch_count", value=0)
    print("No new rows to write. Exiting cleanly.")
    dbutils.notebook.exit("NO_NEW_DATA")


# ----------------------------
# Write / Merge into Silver
# ----------------------------
# Partition by event_date for query performance
if mode == "full" or not delta_path_exists(silver_path):
    (
        silver_batch.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")   # IMPORTANT: replace schema
        .partitionBy("event_date")
        .save(silver_path)
    )
    print("Silver written in overwrite mode (full refresh / first run).")
    #dbutils.notebook.exit("FULL_REFRESH_DONE")  # optional but recommended: guarantees no merge

else:
    # MERGE incremental upsert (idempotent)
    delta_silver = DeltaTable.forPath(spark, silver_path)

    # Use keys that uniquely identify an event record in your dataset
    merge_condition = """
      t.user_session = s.user_session
      AND t.event_ts = s.event_ts
      AND t.event_type = s.event_type
      AND t.product_id = s.product_id
    """

    (
        delta_silver.alias("t")
        .merge(silver_batch.alias("s"), merge_condition)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("Silver MERGE completed (incremental).")


# ----------------------------
# Pass impacted dates to Gold (for efficient recompute)
# ----------------------------
impacted_dates = (
    silver_batch
    .select("event_date")
    .distinct()
    .orderBy("event_date")
    .limit(5000)  # safety
    .toPandas()["event_date"]
    .astype(str)
    .tolist()
)

impacted_dates_str = ",".join(impacted_dates)

dbutils.jobs.taskValues.set(key="impacted_dates", value=impacted_dates_str)
dbutils.jobs.taskValues.set(key="silver_batch_count", value=int(batch_count))

print(f"Impacted dates sent to Gold: {impacted_dates_str[:200]}{'...' if len(impacted_dates_str) > 200 else ''}")

